# House Price Prediction — Project 3
**CloudExify Summer Internship 2026 — Data Science Month 2**

Predicting house prices from area, bedrooms, age, and location using Linear Regression and Random Forest.

> Works in both **Jupyter (local)** and **Google Colab**. If running in Colab, the next cell
> will prompt you to upload `sample_data.csv` automatically.


## Step 0 — Upload data (Colab only)

In [ ]:
# This cell only does something when running in Google Colab.
# In local Jupyter, it does nothing and is safe to run.
import sys

if 'google.colab' in sys.modules:
    from google.colab import files
    print("Please upload sample_data.csv:")
    uploaded = files.upload()
else:
    print("Not running in Colab — make sure sample_data.csv is in the same folder as this notebook.")


## Step 1 — Load and Explore Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('sample_data.csv')

# Explore
print(df.head())
print()
print(df.describe())
print()
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Columns:", list(df.columns))


## Step 2 — Prepare Data

In [ ]:
# Handle missing values (Age had a few missing entries)
df = df.dropna()
print(f"Rows after dropping missing values: {len(df)}")


### Handle outliers
A few listings have unusually high or low prices (e.g. data entry errors or luxury outliers).
We remove rows where `Price` falls outside 1.5x the interquartile range (IQR) — a standard,
defensible way to flag outliers without hand-picking which rows to drop.


In [ ]:
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before = len(df)
df = df[(df['Price'] >= lower_bound) & (df['Price'] <= upper_bound)]
after = len(df)

print(f"Price outlier bounds: Rs {lower_bound:,.0f} to Rs {upper_bound:,.0f}")
print(f"Removed {before - after} outlier row(s); {after} rows remain")


In [ ]:
# Select features (X) and target (y)
X = df[['Area', 'Bedrooms', 'Age', 'Location']]
y = df['Price']

# Convert categorical column (Location) to numeric via one-hot encoding
X = pd.get_dummies(X)
print("Feature columns after encoding:", list(X.columns))

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)}")
print(f"Test set: {len(X_test)}")


## Step 3 — Train Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_train = lr_model.predict(X_train)
y_pred_test = lr_model.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"R² (Train): {r2_train:.3f}")
print(f"R² (Test): {r2_test:.3f}")
print(f"RMSE: Rs {rmse_test:,.0f}")
print(f"MAE: Rs {mae_test:,.0f}")


## Step 4 — Random Forest (usually a better model)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Random Forest R²: {r2_rf:.3f}")
print(f"Random Forest RMSE: Rs {rmse_rf:,.0f}")
print(f"Random Forest MAE: Rs {mae_rf:,.0f}")

# Compare models
if r2_rf > r2_test:
    print("\nRandom Forest is better!")
    best_model = rf_model
else:
    print("\nLinear Regression is better!")
    best_model = lr_model


## Step 5 — Feature Importance

In [ ]:
importances = rf_model.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='#2E86AB')
plt.xlabel('Importance')
plt.title('Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Step 6 — Make Predictions on New Data

In [ ]:
# Predict price for a new house
# Note: dummy columns must match the training columns exactly
new_house = pd.DataFrame({
    'Area': [350],
    'Bedrooms': [4],
    'Age': [10],
    'Location_East': [0],
    'Location_North': [1],
    'Location_South': [0]
})

# Ensure column order matches training data
new_house = new_house[X.columns]

predicted_price = best_model.predict(new_house)
print(f"Predicted price for new house: Rs {predicted_price[0]:,.0f}")


## Try Your Own Prediction

Edit the values below and re-run this cell to get a price prediction — no extra
setup or extensions needed (unlike interactive sliders, which need
`ipywidgets` to be properly configured in your Jupyter environment).


In [ ]:
# ---- EDIT THESE VALUES ----
my_area = 300          # in sqft
my_bedrooms = 3
my_age = 5              # in years
my_location = 'North'   # 'North', 'South', or 'East'
# ----------------------------

row = pd.DataFrame({
    'Area': [my_area],
    'Bedrooms': [my_bedrooms],
    'Age': [my_age],
    'Location_East': [1 if my_location == 'East' else 0],
    'Location_North': [1 if my_location == 'North' else 0],
    'Location_South': [1 if my_location == 'South' else 0],
})
row = row[X.columns]

pred = best_model.predict(row)[0]
print(f"Predicted price: Rs {pred:,.0f}")


### Optional — Interactive widget version
If you'd like sliders and a button instead, install and enable ipywidgets first:
```
pip install --upgrade ipywidgets widgetsnbextension
jupyter nbextension enable --py widgetsnbextension
```
Then restart the kernel and hard-refresh the browser before running the cell below.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

area_slider = widgets.IntSlider(value=250, min=50, max=500, step=10, description='Area (sqft):', style={'description_width': 'initial'})
bedrooms_slider = widgets.IntSlider(value=3, min=1, max=5, step=1, description='Bedrooms:', style={'description_width': 'initial'})
age_slider = widgets.IntSlider(value=10, min=0, max=39, step=1, description='Age (years):', style={'description_width': 'initial'})
location_dropdown = widgets.Dropdown(options=['North', 'South', 'East'], value='North', description='Location:', style={'description_width': 'initial'})
predict_button = widgets.Button(description='Predict Price', button_style='success')
output = widgets.Output()

def on_predict_click(b):
    with output:
        clear_output()
        row = pd.DataFrame({
            'Area': [area_slider.value],
            'Bedrooms': [bedrooms_slider.value],
            'Age': [age_slider.value],
            'Location_East': [1 if location_dropdown.value == 'East' else 0],
            'Location_North': [1 if location_dropdown.value == 'North' else 0],
            'Location_South': [1 if location_dropdown.value == 'South' else 0],
        })
        row = row[X.columns]
        pred = best_model.predict(row)[0]
        print(f"Predicted price: Rs {pred:,.0f}")

predict_button.on_click(on_predict_click)

display(area_slider, bedrooms_slider, age_slider, location_dropdown, predict_button, output)


## Visualization — Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.6, color='#2E86AB', label='Random Forest predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Price (Rs)')
plt.ylabel('Predicted Price (Rs)')
plt.title('Actual vs Predicted House Prices (Random Forest)')
plt.legend()
plt.tight_layout()
plt.show()


## Summary

| Model | R² (Test) | RMSE |
|---|---|---|
| Linear Regression | printed above | printed above |
| Random Forest | printed above | printed above |

The model with the higher R² and lower RMSE on the test set was selected as `best_model` and used to predict the price of a new house in Step 6.
